# Phase 3C.01 — Signal Construction

## Research Objective

Construct a research dataset to evaluate the predictive ability of Time-Series Momentum across 16 ETFs. The session covers data preparation, momentum calculation, directional signals and alignment with forward returns.

**Scope:** Train (2010–2017) and Validation (2018–2021). Test (2022–2025) is excluded.

## 1. Environment and Imports

Import libraries and reusable functions from the project's Python modules.

In [3]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

## 2. Data Loading

Retrieve adjusted ETF closing prices using `download_prices()` in `src/data.py`. History begins in 2008 to initialize the first signals for 2010; data end in December 2021 to preserve the Test sample.

In [4]:
from src.data import download_prices

asset_universe = {
    "Equities" : ["SPY", "VGK", "EWJ", "EEM"],
    "Commodities" : ["USO", "DBA", "DBB", "DBC"],
    "Bonds" : ["IEF", "TLT", "LQD", "HYG"],
    "Forex" : ["FXE", "FXY", "FXB", "FXA"]
}

tickers = []

for assets in asset_universe.values():
    tickers.extend(assets)

print(tickers)

prices = download_prices(tickers, "2008-01-01", "2022-01-01")

print(prices.head())
print(prices.shape)
print(prices.columns)

['SPY', 'VGK', 'EWJ', 'EEM', 'USO', 'DBA', 'DBB', 'DBC', 'IEF', 'TLT', 'LQD', 'HYG', 'FXE', 'FXY', 'FXB', 'FXA']
Ticker            DBA        DBB        DBC        EEM        EWJ        FXA  \
Date                                                                           
2008-01-02  28.197397  18.034828  26.991880  33.408066  38.190754  64.639450   
2008-01-03  28.863743  18.908262  27.199959  33.708656  38.335106  64.434654   
2008-01-04  29.167400  18.455372  27.041822  32.696156  37.353622  63.754559   
2008-01-07  28.779396  18.544334  26.550756  32.935707  37.122681  63.761898   
2008-01-08  29.099916  19.207495  26.792124  32.680328  37.036079  64.456604   

Ticker             FXB         FXE        FXY        HYG        IEF  \
Date                                                                  
2008-01-02  172.158768  134.518723  91.190002  31.210009  55.187603   
2008-01-03  171.550690  134.884109  91.400002  31.150402  55.300949   
2008-01-04  171.515976  134.957169  92.110

## 3. Data Validation

Check downloaded data for temporal coverage, dimensions, missing values, duplicate dates and nonpositive prices before research use.

In [31]:
print(prices.shape) # Dimensions du DataFrame
print(prices.index.min()) # Première date disponible
print(prices.index.max()) # Dernière date disponible
print(len(prices.columns)) # Nombre d'ETF
print(prices.isna().sum().sum()) # Nombre de valeurs manquantes par actifs
print(prices.index.duplicated().sum()) # Nombre de dates dupliquées
print((prices <= 0).sum().sum()) # Nombre de prix inferieur ou égale a zéro

(3526, 16)
2008-01-02 00:00:00
2021-12-31 00:00:00
16
0
0
0


### Conclusion — Data Validation

The universe includes 16 ETFs across four asset classes. Data span January 2008–December 2021, including the required warm-up history. No missing values, duplicate dates or nonpositive prices were detected. Test (2022–2025) remains excluded.

## 4. Monthly Data Preparation

Convert daily prices to month-end prices using `to_monthly_prices()` in `src/data.py`, consistent with the monthly momentum protocol.

In [6]:
import importlib
import src.data

importlib.reload(src.data) # On force la relecture de src.data car le programme ne trouvais pas to_monthly_prices dans src.data car ajouter après le premier import.

from src.data import to_monthly_prices

monthly_prices = to_monthly_prices(prices)
print(monthly_prices.head())
print(monthly_prices.shape)
print(monthly_prices.index.min())
print(monthly_prices.index.max())
print(monthly_prices.isna().sum())
print(monthly_prices.index.duplicated().sum())

Ticker            DBA        DBB        DBC        EEM        EWJ        FXA  \
Date                                                                           
2008-01-31  31.200176  19.611866  27.033497  30.937828  36.718555  65.955803   
2008-02-29  35.054855  21.755014  30.046467  31.554815  36.170071  68.886520   
2008-03-31  30.744696  20.388252  29.771803  30.370548  35.708210  67.821289   
2008-04-30  30.871216  20.388252  31.569597  33.145897  38.335106  70.361115   
2008-05-31  29.816870  19.385418  33.750256  34.194565  39.085625  71.783897   

Ticker             FXB         FXE         FXY        HYG        IEF  \
Date                                                                   
2008-01-31  173.548599  136.327133   93.910004  31.015524  56.636105   
2008-02-29  173.889572  139.494629   96.129997  30.435738  57.337982   
2008-03-31  174.371033  145.382828  100.239998  30.723690  58.104740   
2008-04-30  175.323502  144.189972   95.949997  31.767326  56.700089   
2008-05

### Conclusion — Monthly Data Preparation

The month-end dataset contains **168 monthly observations and 16 ETFs**, from January 2008 to December 2021. No missing values or duplicate dates were detected after resampling. The data are ready for TSMOM signal construction.

## 5. Momentum Calculation

Calculate historical cumulative returns with `calculate_momentum()` in `src/signals.py`. $P_{i,t}$ is the adjusted month-end price of asset $i$ and $L$ is the lookback in months. The primary horizon is 12 months; 3 and 6 months are robustness alternatives.

$$
M_{i,t}^{(L)}=\frac{P_{i,t}}{P_{i,t-L}}-1
$$

**Output:** $M_{i,t}^{(L)}$ measures the price return over the preceding $L$ months; the implemented $L=12$ result is `momentum_12m`. The alternative horizons are evaluated in 3C.03.

In [7]:
import importlib
import src.signals

importlib.reload(src.signals)

from src.signals import calculate_momentum

momentum_12m = calculate_momentum(monthly_prices, lookback=12)
print(momentum_12m.head(24))

Ticker           DBA       DBB       DBC       EEM       EWJ       FXA  \
Date                                                                     
2008-01-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-02-29       NaN       NaN       NaN       NaN       NaN       NaN   
2008-03-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-04-30       NaN       NaN       NaN       NaN       NaN       NaN   
2008-05-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-06-30       NaN       NaN       NaN       NaN       NaN       NaN   
2008-07-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-08-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-09-30       NaN       NaN       NaN       NaN       NaN       NaN   
2008-10-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-11-30       NaN       NaN       NaN       NaN       NaN       NaN   
2008-12-31       NaN       NaN       N

### 5.1. Momentum Consistency Check

Verify the 12-month cumulative return for SPY on January 31, 2009 against the direct calculation from January 2008 and January 2009 prices. Both calculations must agree to numerical precision.

In [8]:
prices_start = monthly_prices.loc["2008-01-31", "SPY"]
prices_end = monthly_prices.loc["2009-01-31", "SPY"]

print(prices_start)
print(prices_end)

manual_momentum = (prices_end / prices_start) - 1 
print(manual_momentum)
print(momentum_12m["SPY"].loc["2009-01-31"])
sanity_check = manual_momentum - momentum_12m["SPY"].loc["2009-01-31"]
sanity_check

97.60250854492188
60.26791000366211
-0.382516792835057
-0.382516792835057


np.float64(0.0)

### Conclusion — Momentum Calculation

`calculate_momentum()` computes cumulative returns over a configurable historical window. SPY's 12-month momentum on January 31, 2009 was independently verified: manual −38.25%, function −38.25%, difference 0. The first 12 observations remain missing during warm-up.

## 6. Directional Signal Construction

Convert 12-month momentum into +1 (positive), −1 (negative) or 0 (neutral), preserving missing observations.

$$
s_{i,t}^{(L)}=\operatorname{sign}(M_{i,t}^{(L)})
$$

**Notation and implementation.** $s_{i,t}^{(L)}$ is the direction of the historical momentum $M_{i,t}^{(L)}$, not a portfolio weight. `signals_12m` stores $s_{i,t}^{(12)}$.

In [9]:
import importlib
import src.signals

importlib.reload(src.signals)

from src.signals import generate_signals

signals_12m = generate_signals(momentum_12m)

print(signals_12m.head())
print(signals_12m.loc["2009-01-31"])

Ticker      DBA  DBB  DBC  EEM  EWJ  FXA  FXB  FXE  FXY  HYG  IEF  LQD  SPY  \
Date                                                                          
2008-01-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-02-29  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-03-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-04-30  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-05-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   

Ticker      TLT  USO  VGK  
Date                       
2008-01-31  NaN  NaN  NaN  
2008-02-29  NaN  NaN  NaN  
2008-03-31  NaN  NaN  NaN  
2008-04-30  NaN  NaN  NaN  
2008-05-31  NaN  NaN  NaN  
Ticker
DBA   -1.0
DBB   -1.0
DBC   -1.0
EEM   -1.0
EWJ   -1.0
FXA   -1.0
FXB   -1.0
FXE   -1.0
FXY    1.0
HYG   -1.0
IEF    1.0
LQD   -1.0
SPY   -1.0
TLT    1.0
USO   -1.0
VGK   -1.0
Name: 2009-01-31 00:00:00, dtype: float64


### Conclusion — Signal Construction

`generate_signals()` converts momentum into directional predictions via `np.sign()`. Warm-up NaNs remain intact. Signals use only information available at the formation date.

## 7. Forward Return Construction

Calculate the following month's realized return and index it by the signal formation date $t$.

$$
r_{i,t+1}=\frac{P_{i,t+1}}{P_{i,t}}-1
$$

Forward returns are for predictive evaluation, not signal construction.

**Notation and implementation.** $P_{i,t}$ and $P_{i,t+1}$ are consecutive adjusted month-end prices of ETF $i$. $r_{i,t+1}$ is the next-month realized return, stored in `forward_returns_1m` at the formation date $t$.

In [10]:
forward_returns_1m = (monthly_prices.pct_change()).shift(-1)
print(forward_returns_1m.head())
print(forward_returns_1m.tail())

prices_january = monthly_prices.loc["2008-01-31", "SPY"]
prices_february = monthly_prices.loc["2008-02-29", "SPY"]

manual_forward_returns_1m = (prices_february / prices_january) -1
print(manual_forward_returns_1m)
verification = (forward_returns_1m.loc["2008-01-31", "SPY"] - manual_forward_returns_1m)
print(verification)

Ticker           DBA       DBB       DBC       EEM       EWJ       FXA  \
Date                                                                     
2008-01-31  0.123547  0.109278  0.111453  0.019943 -0.014938  0.044435   
2008-02-29 -0.122955 -0.062825 -0.009141 -0.037530 -0.012769 -0.015464   
2008-03-31  0.004115  0.000000  0.060386  0.091383  0.073566  0.037449   
2008-04-30 -0.034153 -0.049187  0.069075  0.031638  0.019578  0.020221   
2008-05-31  0.150778  0.043388  0.104316 -0.092695 -0.074608  0.008131   

Ticker           FXB       FXE       FXY       HYG       IEF       LQD  \
Date                                                                     
2008-01-31  0.001965  0.023235  0.023640 -0.018693  0.012393 -0.007063   
2008-02-29  0.002769  0.042211  0.042755  0.009461  0.013373 -0.005971   
2008-03-31  0.005462 -0.008205 -0.042797  0.033968 -0.024174  0.012447   
2008-04-30  0.000467 -0.000179 -0.013549 -0.005084 -0.017774 -0.022052   
2008-05-31  0.008563  0.014124 -0.007

### Conclusion — Forward Return Construction

One-month forward returns use `.pct_change().shift(-1)`. For SPY on January 31, 2008, the computed and manual values both equal −2.5842%, with zero difference. Calculation and alignment are validated. The last observation is missing because the following month's price is unavailable.

## 8. Directional Target Construction

Transform `forward_returns_1m` into `targets_1m` using `generate_signals()`.

$$
y_{i,t}=\operatorname{sign}(r_{i,t+1})
$$

Targets are +1 for positive returns, −1 for negative returns, 0 for zero returns and NaN if unavailable. They represent the realized direction of the next month, used to evaluate momentum predictions.

**Output:** $y_{i,t}$ is the realized direction, not the prediction. It is stored in `targets_1m`; $r_{i,t+1}$ is the input from `forward_returns_1m`.

In [11]:
targets_1m = generate_signals(forward_returns_1m)
print(targets_1m)
print(targets_1m.loc["2008-01-31", "SPY"])
print(targets_1m.loc["2021-12-31", "SPY"])

Ticker      DBA  DBB  DBC  EEM  EWJ  FXA  FXB  FXE  FXY  HYG  IEF  LQD  SPY  \
Date                                                                          
2008-01-31  1.0  1.0  1.0  1.0 -1.0  1.0  1.0  1.0  1.0 -1.0  1.0 -1.0 -1.0   
2008-02-29 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0  1.0  1.0  1.0  1.0  1.0 -1.0 -1.0   
2008-03-31  1.0  0.0  1.0  1.0  1.0  1.0  1.0 -1.0 -1.0  1.0 -1.0  1.0  1.0   
2008-04-30 -1.0 -1.0  1.0  1.0  1.0  1.0  1.0 -1.0 -1.0 -1.0 -1.0 -1.0  1.0   
2008-05-31  1.0  1.0  1.0 -1.0 -1.0  1.0  1.0  1.0 -1.0 -1.0  1.0 -1.0 -1.0   
...         ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...   
2021-08-31 -1.0 -1.0  1.0 -1.0  1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0   
2021-09-30  1.0  1.0  1.0  1.0 -1.0  1.0  1.0 -1.0 -1.0 -1.0 -1.0  1.0  1.0   
2021-10-31  1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0  1.0 -1.0  1.0 -1.0 -1.0   
2021-11-30  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0 -1.0  1.0 -1.0 -1.0  1.0   
2021-12-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  

### Conclusion — Target Construction

Forward returns were transformed into +1, −1, 0 or NaN through `generate_signals()`. The SPY consistency check confirms the expected results.

## 9. Data Alignment Validation

Verify that `signals_12m` and `targets_1m` have identical dimensions, dates and ordered columns, so each prediction is associated with the same ETF and formation date. Correct alignment is essential to avoid methodological errors.

In [12]:
print(signals_12m.shape)
print(targets_1m.shape)

print(signals_12m.index.equals(targets_1m.index))
print(signals_12m.columns.equals(targets_1m.columns))

(168, 16)
(168, 16)
True
True


### Conclusion — Data Alignment

`signals_12m` and `targets_1m` have identical 168 × 16 dimensions, dates and columns. Structural and temporal alignment is validated; the data are ready for predictive sample preparation.

## 10. Train & Validation Sample Preparation

Separate the data chronologically according to the Phase 3B experimental protocol.

### 10.1. Temporal Split — Train & Validation

Samples are defined by realized return months, not signal formation dates:

- Train: January 2010–December 2017 (96 months).
- Validation: January 2018–December 2021 (48 months).
- Test: 2022–2025, reserved for Phase 3F.

Signals are formed one month earlier. Check sample dimensions, signal-target alignment and the absence of overlapping dates.

In [32]:
train_signals = signals_12m.loc["2009-12-31":"2017-11-30"]
train_targets = targets_1m.loc["2009-12-31":"2017-11-30"]

print(train_signals.shape)
print(train_targets.shape)
print(train_signals.index.equals(train_targets.index))

validation_signals = signals_12m.loc["2017-12-31":"2021-11-30"]
validation_targets = targets_1m.loc["2017-12-31":"2021-11-30"]

print(validation_signals.shape)
print(validation_targets.shape)
print(validation_signals.index.equals(validation_targets.index))

dates_verif = (len(train_signals.index.intersection(validation_signals.index)))
print(dates_verif)

print(train_signals.isna().sum().sum())
print(train_targets.isna().sum().sum())
print(validation_signals.isna().sum().sum())
print(validation_targets.isna().sum().sum())

(96, 16)
(96, 16)
True
(48, 16)
(48, 16)
True
0
0
0
0
0


#### Conclusion — Temporal Split

Train contains 96 × 16 observations and Validation 48 × 16. Signal-target alignment is validated and the samples share no dates. The chronological split is consistent with the protocol but does not by itself eliminate all temporal biases.

### 10.2. Data Quality and Final Sample Preparation

Identify missing signals or targets, neutral signals, zero forward returns and valid binary classification observations.

$$
m_{i,t}=\mathbb{1}\{s_{i,t}\in\{-1,+1\},\ y_{i,t}\in\{-1,+1\}\},\qquad N_{\mathrm{valid}}=\sum_i\sum_t m_{i,t}
$$

Document exclusions to preserve research transparency.

**Notation and implementation.** $\mathbb{1}$ equals one when its condition is true and zero otherwise. $s_{i,t}$ and $y_{i,t}$ denote the signal and target; $m_{i,t}$ is the Boolean validity indicator (`train_mask` or `validation_mask`). $N_{\mathrm{valid}}$ is the sum of the mask across ETFs and dates, evaluated separately for each sample.

In [38]:
print((train_signals == 0).sum().sum())
print((train_targets == 0).sum().sum())
print((validation_signals == 0).sum().sum())
print((validation_targets == 0).sum().sum())

valid_train_signals = ((train_signals != 0) & (train_signals.notna()))
valid_train_targets = ((train_targets != 0) & (train_targets.notna()))
train_mask = valid_train_signals & valid_train_targets

valid_validation_signals = ((validation_signals != 0) & (validation_signals.notna()))
valid_validation_targets = ((validation_targets != 0) & (validation_targets.notna()))
validation_mask = valid_validation_signals & valid_validation_targets

print(train_mask.shape)
print(train_mask.sum().sum())

print(validation_mask.shape)
print(validation_mask.sum().sum())



1
4
0
3
(96, 16)
1531
(48, 16)
765


#### Conclusion — Data Quality and Final Sample Preparation

| Indicator | Train | Validation |
|---|---:|---:|
| Initial observations | 1,536 | 768 |
| Missing values | 0 | 0 |
| Neutral signals | 1 | 0 |
| Zero targets | 4 | 3 |
| Valid observations | 1,531 | 765 |

Boolean masks identify pairs that are nonzero and nonmissing without deleting observations from the original DataFrames. The samples are ready for final predictive preparation.

## 11. Final Predictive Dataset

Apply the validity masks using `.where()` to retain only eligible observations while preserving original dimensions, dates and columns. Test remains excluded.

In [ ]:
train_signals_clean = train_signals.where(train_mask)
train_targets_clean = train_targets.where(train_mask)

print(train_signals_clean.shape)
print(train_signals_clean.notna().sum().sum())
print(train_targets_clean.shape)
print(train_targets_clean.notna().sum().sum())

validation_signals_clean = validation_signals.where(validation_mask)
validation_targets_clean = validation_targets.where(validation_mask)

print(validation_signals_clean.shape)
print(validation_signals_clean.notna().sum().sum())
print(validation_targets_clean.shape)
print(validation_targets_clean.notna().sum().sum())

print(train_signals_clean.notna().equals(train_targets_clean.notna()))
print(validation_signals_clean.notna().equals(validation_targets_clean.notna()))

(96, 16)
1531
(96, 16)
1531
(48, 16)
765
(48, 16)
765
True
True


### Conclusion — Final Predictive Dataset

Masks were applied to signals and targets using `.where()` without changing the original DataFrames.

| Indicator | Train | Validation |
|---|---:|---:|
| Initial observations | 1,536 | 768 |
| Valid observations | 1,531 | 765 |
| Excluded observations | 5 | 3 |
| Valid observation alignment | Passed | Passed |

Signals and targets have matching dimensions, dates and nonmissing positions. Exclusions remain identifiable as NaN. The dataset is ready for Phase 3C.02.

## 12. General Conclusion — Phase 3C.01

### Objectives Achieved

The session constructed a predictive research dataset by loading and validating 16 ETF price series, resampling to monthly frequency, calculating 12-month momentum, constructing directional signals and next-month returns and targets, splitting Train and Validation, handling invalid observations and validating alignment.

### Results

There are **1,531 valid Train observations** and **765 valid Validation observations**. Signals use information known at the prediction date and targets represent next-month outcomes. Test (2022–2025) has not been used in this phase.

### Next Step — Phase 3C.02

Evaluate Accuracy, Balanced Accuracy, class-specific Recall, confusion matrices and naive baselines. No conclusion about predictive or economic performance is drawn at this stage.